# Sesión 06 - Momentos de dos o más variables

Objetivo: calcular covarianza, correlación, matriz de covarianza y verificar Cauchy-Schwarz.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(31)
pd.set_option("display.precision", 4)


## 1. Simulación de variables correlacionadas


### Lectura matemática

- **Distribución asumida:** normal multivariada con vector de medias y matriz de covarianza.
- **Parámetros estimados:** covarianza y correlación.
- **Supuesto que puede fallar:** dependencia no lineal o colas pesadas.
- **Diagnóstico:** scatterplot, matriz de covarianza positiva semidefinida y correlaciones robustas.


In [ ]:
media = np.array([100, 70])
cov = np.array([[15 ** 2, -0.65 * 15 * 10], [-0.65 * 15 * 10, 10 ** 2]])
datos = rng.multivariate_normal(media, cov, size=1_500)

df = pd.DataFrame(datos, columns=["precio", "demanda"])
df.head()


In [ ]:
cov_emp = df.cov(ddof=0)
corr_emp = df.corr()

print("Matriz de covarianza")
print(cov_emp)
print("\nMatriz de correlación")
print(corr_emp)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(df["precio"], df["demanda"], alpha=0.35, s=12)
ax.set_title("Relación precio-demanda")
ax.set_xlabel("precio")
ax.set_ylabel("demanda")
plt.show()


## 2. Verificación de Cauchy-Schwarz

Para variables centradas: $|Cov(X,Y)| \le \sigma_X\sigma_Y$.


In [ ]:
x = df["precio"].to_numpy()
y = df["demanda"].to_numpy()

cov_xy = np.mean((x - x.mean()) * (y - y.mean()))
sx = x.std(ddof=0)
sy = y.std(ddof=0)

print(f"|Cov(X,Y)| = {abs(cov_xy):.4f}")
print(f"sigma_X * sigma_Y = {sx * sy:.4f}")
print(f"Se cumple: {abs(cov_xy) <= sx * sy}")
print(f"correlación = {cov_xy / (sx * sy):.4f}")


## 3. Tres o más variables: matriz de covarianza y PCA


In [ ]:
n = 1_500
precio = rng.normal(100, 12, n)
descuento = rng.uniform(0, 0.30, n)
publicidad = rng.normal(50, 10, n)
demanda = 160 - 0.7 * precio + 80 * descuento + 0.5 * publicidad + rng.normal(0, 8, n)

multi = pd.DataFrame(
    {
        "precio": precio,
        "descuento": descuento,
        "publicidad": publicidad,
        "demanda": demanda,
    }
)

multi.cov(ddof=0)


In [ ]:
X_std = StandardScaler().fit_transform(multi)
pca = PCA()
pca.fit(X_std)

pd.DataFrame(
    {
        "componente": np.arange(1, len(pca.explained_variance_ratio_) + 1),
        "varianza_explicada": pca.explained_variance_ratio_,
        "acumulado": np.cumsum(pca.explained_variance_ratio_),
    }
)


## 4. Autocovarianza como momento conjunto temporal


### Lectura matemática

- **Proceso asumido:** AR(1), $X_t=\phi X_{t-1}+arepsilon_t$.
- **Parámetro estimado:** autocovarianza/autocorrelación por rezago.
- **Supuesto que puede fallar:** no estacionariedad, tendencia o estacionalidad.
- **Diagnóstico:** ACF, prueba ADF si está disponible y residuales.


In [ ]:
phi = 0.75
ruido = rng.normal(0, 1, 400)
serie = np.zeros_like(ruido)
for t in range(1, len(serie)):
    serie[t] = phi * serie[t - 1] + ruido[t]

def autocovarianza(x, lag):
    x = np.asarray(x)
    return np.mean((x[lag:] - x.mean()) * (x[:-lag] - x.mean()))

acovs = pd.Series({lag: autocovarianza(serie, lag) for lag in range(1, 11)}, name="autocovarianza")
acovs


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(serie)
axes[0].set_title("Serie AR(1) simulada")
axes[1].bar(acovs.index, acovs.values)
axes[1].set_title("Autocovarianza por rezago")
axes[1].set_xlabel("lag")
plt.show()


## 5. Momentos conjuntos con datos de demanda y tráfico

Este bloque reutiliza `sales_data.csv` para matriz de covarianza/correlación y `resultados_futuro/` para autocovarianza de duraciones de tráfico.


In [ ]:
from pathlib import Path

def display(obj):
    try:
        from IPython.display import display as ipy_display
        ipy_display(obj)
    except Exception:
        if hasattr(obj, "to_string"):
            print(obj.to_string())
        else:
            print(obj)

def encontrar_data_dir():
    for candidato in [Path("data_sources"), Path("../data_sources")]:
        if candidato.exists():
            return candidato
    return None

DATA_DIR = encontrar_data_dir()
if DATA_DIR is None or not (DATA_DIR / "sales_data.csv").exists():
    print("No se encontró data_sources/sales_data.csv. Se mantiene la sección sintética.")
else:
    cols = ["Demand", "Units Sold", "Price", "Discount", "Promotion", "Competitor Pricing"]
    ventas_df = pd.read_csv(DATA_DIR / "sales_data.csv", usecols=cols)
    display(ventas_df.cov(numeric_only=True))
    display(ventas_df.corr(numeric_only=True))

    x = ventas_df["Price"].to_numpy()
    y = ventas_df["Demand"].to_numpy()
    cov_xy = np.mean((x - x.mean()) * (y - y.mean()))
    limite = x.std(ddof=0) * y.std(ddof=0)
    print(f"|Cov(Price,Demand)| = {abs(cov_xy):.4f}")
    print(f"sigma_Price * sigma_Demand = {limite:.4f}")
    print(f"Cauchy-Schwarz: {abs(cov_xy) <= limite}")


In [ ]:
if DATA_DIR is not None and (DATA_DIR / "resultados_futuro").exists():
    archivos = sorted((DATA_DIR / "resultados_futuro").glob("*.csv"))
    trafico = pd.concat([pd.read_csv(a, parse_dates=["timestamp_local"]) for a in archivos], ignore_index=True)
    trafico = trafico.sort_values("timestamp_local")
    serie_trafico = trafico["duracion_en_trafico_min"].dropna().to_numpy()
    acov_trafico = pd.Series({lag: autocovarianza(serie_trafico, lag) for lag in range(1, 13)}, name="autocovarianza")
    display(acov_trafico)


## 6. Series de tiempo: AR(1), ARIMA, ARIMAX y VAR

Este bloque recupera la parte de series de tiempo del legacy. La conexión conceptual es que la autocovarianza describe dependencia de una variable con su propio pasado, y las matrices de covarianza cruzada describen transmisión entre series.


### Lectura matemática

- **Modelos asumidos:** ARIMA para una serie, ARIMAX con covariables y VAR para varias series.
- **Parámetros estimados:** coeficientes autorregresivos, medias/choques y efectos exógenos.
- **Supuesto que puede fallar:** estacionariedad tras diferenciación o covariables endógenas.
- **Diagnóstico:** ADF, ACF/PACF, Ljung-Box y estabilidad de VAR.


In [ ]:
if DATA_DIR is None or not (DATA_DIR / "sales_data.csv").exists():
    print("No se encontró sales_data.csv. Se omite el bloque de series con datos reales.")
else:
    ventas_ts = pd.read_csv(DATA_DIR / "sales_data.csv", parse_dates=["Date"])
    serie_producto = (
        ventas_ts[(ventas_ts["Store ID"] == "S001") & (ventas_ts["Product ID"] == "P0001")]
        .sort_values("Date")
        .set_index("Date")
    )
    y_demanda = serie_producto["Demand"].asfreq("D").interpolate()
    y_lag = y_demanda.shift(1).dropna()
    y_now = y_demanda.loc[y_lag.index]
    phi_ols = np.cov(y_now, y_lag, ddof=0)[0, 1] / np.var(y_lag, ddof=0)
    intercept = y_now.mean() - phi_ols * y_lag.mean()
    resid = y_now - (intercept + phi_ols * y_lag)
    print(f"AR(1) por momentos/OLS: demanda_t = {intercept:.2f} + {phi_ols:.3f} demanda_t-1")
    print(f"Autocorrelación lag 1: {y_demanda.autocorr(lag=1):.3f}")
    print(f"Desv. residual: {resid.std(ddof=1):.2f}")


In [ ]:
try:
    from statsmodels.tsa.stattools import adfuller, acf
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    from statsmodels.tsa.api import VAR
    statsmodels_ok = True
except Exception as exc:
    statsmodels_ok = False
    print("statsmodels no disponible. Instalar con `pip install statsmodels` para ARIMA/ARIMAX/VAR.")
    print(type(exc).__name__, exc)

if DATA_DIR is not None and (DATA_DIR / "sales_data.csv").exists() and statsmodels_ok:
    adf_stat, adf_pvalue, *_ = adfuller(y_demanda)
    print(f"ADF p-value demanda P0001/S001: {adf_pvalue:.4f}")

    arima = SARIMAX(y_demanda, order=(1, 1, 1), enforce_stationarity=False, enforce_invertibility=False)
    arima_res = arima.fit(disp=False)
    print("ARIMA(1,1,1) parámetros")
    display(arima_res.params.to_frame("valor"))

    exog = serie_producto[["Price", "Promotion"]].asfreq("D").interpolate()
    arimax = SARIMAX(y_demanda, exog=exog, order=(1, 1, 1), enforce_stationarity=False, enforce_invertibility=False)
    arimax_res = arimax.fit(disp=False)
    print("ARIMAX con Price y Promotion")
    display(arimax_res.params.to_frame("valor"))

    matriz_productos = (
        ventas_ts[(ventas_ts["Store ID"] == "S001") & (ventas_ts["Product ID"].isin(["P0001", "P0002", "P0003"]))]
        .pivot_table(index="Date", columns="Product ID", values="Demand", aggfunc="mean")
        .asfreq("D")
        .interpolate()
    )
    var_data = matriz_productos.diff().dropna()
    var_model = VAR(var_data)
    var_res = var_model.fit(maxlags=5, ic="aic")
    print(f"VAR rezagos seleccionados por AIC: {var_res.k_ar}")
    display(var_res.params.head())


## Práctica

Cambia la correlación precio-demanda o el parámetro `phi` de la serie AR(1). Describe cómo cambian la matriz de covarianza y las autocovarianzas.
